# Knowledge Graph Pipeline1: Proof-of-Concept (without ODKE+)

This notebook implements a pipeline for financial knowledge extraction using FNSPID dataset.

In [2]:
#from huggingface_hub import hf_hub_download
import pandas as pd
import os
# from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List, Type, TypeVar, Generic
import enum
from datetime import datetime
import requests
from bs4 import BeautifulSoup
from google import genai
from google.genai import types
import json
from neo4j import GraphDatabase, Session, Transaction
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

ImportError: cannot import name 'genai' from 'google' (unknown location)

In [ ]:
def join_string(item):
    Date, Article_title, Stock_symbol, Url, Lexrank_summary = item
    final_string = ""
    
    # Check if column has a unique value
    if pd.notna(Date):
        final_string += f"Date: {Date}"
    
    if pd.notna(Article_title):
        if final_string:
            final_string += " | "
        final_string += f"Article Title: {Article_title}"
    
    if pd.notna(Stock_symbol):
        if final_string:
            final_string += " | "
        final_string += f"Stock Symbol: {Stock_symbol}"
    
    if pd.notna(Url):
        if final_string:
            final_string += " | "
        final_string += f"Url: {Stock_symbol}"
    
    if pd.notna(Lexrank_summary):
        if final_string:
            final_string += " | "
        final_string += f"Summary: {Lexrank_summary}"
    
    return final_string

def filtered_text(file_path):
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip'  # Skip problematic lines
    )
    
    # DataFrame processing
    df['Date'] = pd.to_datetime(df['Date'])
    date_time = input("Date time to filter (e.g., 2023-12-01 00:00:00+00:00): ")
    ticker_symbol = input("Ticker symbol to filter (e.g., AAPL, AMZN, GOOGL): ")
    df_filtered = df[(df['Date'].dt.date == pd.to_datetime(date_time).date()) & (df['Stock_symbol'] == ticker_symbol)]
    
    # Create the 'information' column
    df_filtered['Information'] = df_filtered[
        ['Date', 'Article_title', 'Stock_symbol','Lexrank_summary']
    ].apply(join_string, axis=1)
    
    # Group all information and return as text
    df_grouped = df_filtered.groupby('Date')['Information'].apply(lambda x: '\n'.join(x)).reset_index()
    sample = df_grouped.head().loc[0, 0]['Information']
    
    return sample

In [ ]:
def extract_entities_and_relationship(text):
    prompt = f"""
# Role
You are an expert Financial Data Analyst and Knowledge Graph Engineer. Your task is to analyze financial news summaries from Nasdaq and extract a structured knowledge graph in JSON format.

# Task
You will be provided with a raw text block containing metadata (Date, Title, Ticker, URL) and a Summary. You must extract semantic triplets representing the relationships between entities mentioned in the text.

# Ontology (Strict Compliance Required)
You must strictly use ONLY the following Entity Types and Relationship Types. Do not invent new types.

## 1. Entity Types (24 Categories)
**Core Business Entities**
- ORG: Filing Company (the main subject of the news/ticker)
- COMP: External companies (competitors, suppliers, customers, partners)
- SEGMENT: Internal business divisions (e.g., Cloud segment)
- PERSON: Key individuals (Executives, Board members)

**Geographic & Regulatory**
- GPE: Geographic entities (Countries, cities)
- ORG_GOV: Government bodies (e.g., US Gov)
- ORG_REG: Regulatory bodies (SEC, Fed, ECB)

**Financial & Market**
- FIN_INST: Financial instruments (Stocks, bonds, options)
- FIN_MARKET: Market indices (S&P 500, Nasdaq)
- FIN_METRIC: Financial metrics (Revenue, EPS, Share Price)
- ECON_IND: Economic indicators (Inflation, GDP)

**Products & Operations**
- PRODUCT: Products or services (iPhone, AWS)
- CONCEPT: Abstract concepts (AI, Digital Transformation)
- RAW_MATERIAL: Essential materials (Lithium, Oil)
- LOGISTICS: Supply chain entities (Ports, Shipping lanes)

**Risk & Compliance**
- RISK_FACTOR: Documented risks (Recession risk, Cyber attacks)
- LITIGATION: Legal disputes, lawsuits
- REGULATORY_REQUIREMENT: Specific regulations (GDPR, Basel III)
- ACCOUNTING_POLICY: Policies (Revenue recognition)

**Strategic & ESG**
- EVENT: Material events (Earnings call, M&A, Pandemic)
- SECTOR: Industries (Technology, Healthcare)
- ESG_TOPIC: ESG themes (Carbon, DEI)
- MACRO_CONDITION: Economic trends (Recession, Labor shortage)
- COMMENTARY: Management statements/guidance

## 2. Relationship Types (27 Categories)
**Ownership & Control**
- Has_Stake_In
- Regulates
- Operates_In

**Business Activities**
- Announces
- Introduces
- Produces
- Invests_In
- Partners_With
- Supplies

**Financial Impact**
- Impacts
- Positively_Impacts
- Negatively_Impacts
- Increases
- Decreases
- Affects_Stock

**Risk & Events**
- Involved_In
- Impacted_By
- Faces
- Depends_On

**Market & Reporting**
- Discloses
- Guides_On
- Complies_With
- Subject_To

**Additional Relations**
- Related_To
- Member_Of
- Causes_Shortage_Of
- Stock_Decline_Due_To
- Stock_Rise_Due_To
- Market_Reacts_To

# Extraction Rules
1. **Metadata Extraction:** Parse the input header to extract the global `date`, `ticker`, and `source_url`. Apply these to every triplet found in that text.
2. **Entity Resolution:**
   - If the main ticker (e.g., AAPL) is mentioned, label it as `ORG`.
   - Other companies mentioned (e.g., McDonald's in an Apple article) should be labeled `COMP`.
3. **Granularity:** Extract specific named entities from the text (e.g., "Apple Inc." instead of just "Company").
4. **Source Text:** You must include the exact sentence or phrase where the relationship was found in the `source_text` field.
5. **Preserve ALL numerical figures and dates exactly.**
6. **Keep ALL dates in their original format.**

# Output Format
Return a single JSON list of objects. Do not include markdown formatting (like ```json) outside of the list.

JSON Structure:
[
    {{
        "triplet_id": "UUID or Sequence (optional)",
        "entity": {{
            "name": "Extracted Head Entity Name",
            "entity_type": "One of the 24 entity types (ORG, COMP, PERSON, etc.)"
        }},
        "relationship": {{
            "relationship": "One of the 27 relationship types (Produces, Increases, etc.)",
            "evidence": "Evidence text snippet supporting the relationship"
        }},
        "target": {{
            "name": "Extracted Tail Entity Name",
            "entity_type": "One of the 24 entity types (PRODUCT, FIN_METRIC, etc.)"
        }},
        "temporal_info": {{
            "date": "YYYY-MM-DD",
            "extraction_type": "extracted or default"
        }},
        "chunk_text": "Full text context surrounding the triplet (the exact sentence/phrase where relationship was found)",
        "chunk_id": null,
        "page_id": null
    }}
]

Note: For compatibility with existing code, also include these flat fields at the root level:
- "date": "YYYY-MM-DD" (from metadata)
- "ticker": "Symbol from metadata"
- "source_url": "URL from metadata"

# Few-Shot Example

**Input:**
Date: 2023-12-01 | Article Title: Best Blue Chip Stocks? | Stock Symbol: AAPL | Url: https://nasdaq.com/article/123
Summary: Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics. Recently, shares of AAPL stock have gained by 9.29% due to holiday sales.

**Output:**
[
    {{
        "triplet_id": "1",
        "entity": {{
            "name": "Apple Inc.",
            "entity_type": "ORG"
        }},
        "relationship": {{
            "relationship": "Produces",
            "evidence": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics."
        }},
        "target": {{
            "name": "consumer electronics",
            "entity_type": "PRODUCT"
        }},
        "temporal_info": {{
            "date": "2023-12-01",
            "extraction_type": "default"
        }},
        "chunk_text": "Apple Inc. (AAPL) is a multinational technology company that specializes in consumer electronics.",
        "chunk_id": null,
        "page_id": null,
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123"
    }},
    {{
        "triplet_id": "2",
        "entity": {{
            "name": "AAPL",
            "entity_type": "ORG"
        }},
        "relationship": {{
            "relationship": "Increases",
            "evidence": "shares of AAPL stock have gained by 9.29%"
        }},
        "target": {{
            "name": "Share Price",
            "entity_type": "FIN_METRIC"
        }},
        "temporal_info": {{
            "date": "2023-12-01",
            "extraction_type": "default"
        }},
        "chunk_text": "shares of AAPL stock have gained by 9.29%",
        "chunk_id": null,
        "page_id": null,
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123"
    }},
    {{
        "triplet_id": "3",
        "entity": {{
            "name": "AAPL",
            "entity_type": "ORG"
        }},
        "relationship": {{
            "relationship": "Stock_Rise_Due_To",
            "evidence": "shares of AAPL stock have gained by 9.29% due to holiday sales"
        }},
        "target": {{
            "name": "Holiday Sales",
            "entity_type": "EVENT"
        }},
        "temporal_info": {{
            "date": "2023-12-01",
            "extraction_type": "default"
        }},
        "chunk_text": "shares of AAPL stock have gained by 9.29% due to holiday sales.",
        "chunk_id": null,
        "page_id": null,
        "date": "2023-12-01",
        "ticker": "AAPL",
        "source_url": "https://nasdaq.com/article/123"
    }}
]

# Input Text for Processing
{text}
"""
    load_dotenv()
    client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY', ''))
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    
    result = json.loads(response.text)
    return result

In [ ]:
def clear_kg(kg):
    with kg._driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")

def add_relationship_to_neo4j(kg, triplets):
    """
    Add relationships to Neo4j graph.
    Handles both nested FinancialTripletModel structure and flat structure for backward compatibility.
    """
    with kg._driver.session() as session:
        for triplet in triplets:
            # Handle nested FinancialTripletModel structure
            if isinstance(triplet.get('entity'), dict):
                # Nested structure (FinancialTripletModel format)
                entity_name = triplet['entity']['name']
                entity_type = triplet['entity']['entity_type']
                rel_type = triplet['relationship']['relationship']
                target_name = triplet['target']['name']
                target_type = triplet['target']['entity_type']
                source_text = triplet.get('chunk_text', triplet['relationship'].get('evidence', ''))
            else:
                # Flat structure (backward compatibility)
                entity_name = triplet['entity']
                entity_type = triplet['entity_type']
                rel_type = triplet['relationship']
                target_name = triplet['target']
                target_type = triplet['target_type']
                source_text = triplet.get('source_text', '')
            
            query = f"MERGE (a:Entity {{name: $source}}) " \
                    f"ON CREATE SET a.type = $source_type " \
                    f"MERGE (b:Entity {{name: $target}}) " \
                    f"ON CREATE SET b.type = $target_type " \
                    f"MERGE (a)-[r:{rel_type}]->(b) " \
                    f"SET r.date = $date, " \
                    f"r.ticker = $ticker, " \
                    f"r.source_url = $source_url, " \
                    f"r.source_text = $source_text"
            try:
                session.run(
                    query,
                    source=entity_name,
                    source_type=entity_type,
                    target=target_name,
                    target_type=target_type,
                    date=triplet.get('date', 'N/A'),
                    ticker=triplet.get('ticker', 'N/A'),
                    source_url=triplet.get('source_url', ''),
                    source_text=source_text
                )
            except Exception as e:
                print(f"Error adding relationship: {e}")                        

In [ ]:
if __name__ == "__main__":
    # 1. Download dataset from huggingface
    # file_path = hf_hub_download(
    #     repo_id="Zihan1004/FNSPID",
    #     filename="Stock_news/nasdaq_exteral_data.csv",
    #     repo_type="dataset"
    # )
    # sample = filtered_text(file_path)
    #with open('/Users/anhvu/Desktop/Submission/IDM/Assignment3/sample-text', 'r') as f: # hard-coded for POC development
    #    sample = f.read()
        
    # 2. Load and filter CSV data for AAPL on 2023-12-01 using financial_ontology
    from financial_ontology import (
        FinancialKnowledgeGraphData,
        FinancialTripletModel,
        EntityModel,
        RelationshipModel,
        EntityTypeEnum,
        RelationshipTypeEnum
    )
    
    # Load CSV and filter for AAPL on 2023-12-01
    file_path = 'nasdaq_100.csv'
    df = pd.read_csv(
        file_path,
        dtype=str,
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='skip'
    )
    
    # Filter for AAPL on 2023-12-01
    df['Date'] = pd.to_datetime(df['Date'])
    target_date = pd.to_datetime('2023-12-01').date()
    df_filtered = df[
        (df['Date'].dt.date == target_date) & 
        (df['Stock_symbol'] == 'AAPL')
    ]
    
    if df_filtered.empty:
        print(f"No data found for AAPL on 2023-12-01")
        triplets = []
    else:
        # Create the 'information' column using join_string function
        df_filtered['Information'] = df_filtered[
            ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Lexrank_summary']
        ].apply(join_string, axis=1)
        
        # Group all information and return as text
        df_grouped = df_filtered.groupby('Date')['Information'].apply(
            lambda x: '\n'.join(x)
        ).reset_index()
        
        if len(df_grouped) > 0:
            sample_text = df_grouped.iloc[0]['Information']
            print(f"Extracted text from {len(df_filtered)} articles for AAPL on 2023-12-01")
            
            # Use LLMs to extract kg entities and relationship
            triplets = extract_entities_and_relationship(sample_text)
            print(f"Extracted {len(triplets)} triplets using financial_ontology structure")
        else:
            print("No grouped data found")
            triplets = []
    
       
    # 3. Add relationships to Neo4j Instance
    load_dotenv()
    graph = Neo4jGraph(
        url=os.getenv('NEO4J_URI', ''),
        username=os.getenv('NEO4J_USERNAME', ''),
        password=os.getenv('NEO4J_PASSWORD', ''),
        database=os.getenv('NEO4J_DATABASE', '')
    )
    clear_kg(graph)
    add_relationship_to_neo4j(graph, triplets)
    
    # 4. LLMs output from few-shot guiding questions (10 questions)
    
    # 5. Run LLM as a judge on input and output, compare with groundtruth answer done by human